# лабораторная работа 6.2
## arima для трех рядов 

в этой работе я беру 3 ряда из yearly-части соревнования m3:
- `n 153`
- `n 156`
- `n 158`

In [1]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.float_format = '{:,.3f}'.format

для 
- историческая часть
- скрытая будущая часть, которая в соревновании использовалась как test

In [ ]:
workbook = pd.ExcelFile('M3C.xls')
yearly_sheet = workbook.parse('M3Year').to_numpy()

yearly_series = {}

for row in yearly_sheet:
    series_name = row[0]
    total_length = int(row[1])
    test_length = int(row[2])
    start_year = int(row[4])

    train_values = pd.to_numeric(
        pd.Series(row[6:6 + total_length - test_length]),
        errors='coerce',
    ).dropna().astype(float).reset_index(drop=True)

    test_values = pd.to_numeric(
        pd.Series(row[6 + total_length - test_length:6 + total_length]),
        errors='coerce',
    ).dropna().astype(float).reset_index(drop=True)

    yearly_series[series_name] = {
        'start_year': start_year,
        'train': train_values,
        'test': test_values,
        'official_horizon': test_length,
    }

selected_series_names = ['N 153', 'N 156', 'N 158']

summary_rows = []
for series_name in selected_series_names:
    info = yearly_series[series_name]
    summary_rows.append(
        {
            'ряд': series_name,
            'стартовый год': info['start_year'],
            'длина train': len(info['train']),
            'длина official test': len(info['test']),
            'горизонт в работе': 4,
        }
    )

selected_summary = pd.DataFrame(summary_rows)
display(selected_summary)

## как я буду подбирать модели

для каждого ряда шаги одинаковые:
- смотрю на исходный график;
- проверяю стационарность через adf;
- строю `acf` и `pacf` для разностей;
- перебираю набор `arima(p,d,q)` с `p,q` от 0 до 3 и `d` от 0 до 2;
- выбираю модель по ошибке прогноза на первых 4 точках official test.

дополнительно смотрю на `aic` и на `ljung-box`, чтобы остатки не выглядели подозрительно.

In [ ]:
def make_adf_table(series_values):
    rows = []

    for row_name, values in [
        ('исходный ряд', series_values),
        ('первая разность, d=1', series_values.diff()),
        ('вторая разность, d=2', series_values.diff().diff()),
    ]:
        clean_values = values.dropna()
        adf_stat, p_value, _, _, _, _ = adfuller(clean_values, autolag='AIC')
        rows.append(
            {
                'ряд': row_name,
                'наблюдений': len(clean_values),
                'adf-статистика': adf_stat,
                'p-value': p_value,
            }
        )

    return pd.DataFrame(rows)


def choose_diff_series(series_values):
    diff_1 = series_values.diff().dropna()
    diff_2 = series_values.diff().diff().dropna()

    p_value_d1 = adfuller(diff_1, autolag='AIC')[1]
    if p_value_d1 < 0.05:
        return diff_1, 1

    return diff_2, 2


def evaluate_orders(train_values, actual_future, p_max=3, d_max=2, q_max=3):
    rows = []
    ljung_box_lag = min(10, max(3, len(train_values) // 3))

    for d in range(d_max + 1):
        for p in range(p_max + 1):
            for q in range(q_max + 1):
                if p == 0 and d == 0 and q == 0:
                    continue

                try:
                    model = ARIMA(
                        train_values,
                        order=(p, d, q),
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    )
                    fit = model.fit()
                    forecast = fit.forecast(steps=len(actual_future))

                    mae = np.mean(np.abs(actual_future.values - forecast.values))
                    rmse = np.sqrt(np.mean((actual_future.values - forecast.values) ** 2))
                    mape = np.mean(
                        np.abs((actual_future.values - forecast.values) / actual_future.values)
                    ) * 100

                    ljung_box_pvalue = acorr_ljungbox(
                        fit.resid.dropna(),
                        lags=[ljung_box_lag],
                        return_df=True,
                    )['lb_pvalue'].iloc[0]

                    rows.append(
                        {
                            'order': (p, d, q),
                            'aic': fit.aic,
                            'bic': fit.bic,
                            'mae': mae,
                            'rmse': rmse,
                            'mape, %': mape,
                            'ljung-box p-value': ljung_box_pvalue,
                            'сошлась': fit.mle_retvals.get('converged', True),
                        }
                    )
                except Exception:
                    pass

    result = pd.DataFrame(rows)
    result = result.sort_values(by=['rmse', 'aic']).reset_index(drop=True)
    return result

## анализ выбранных рядов

ниже для каждого ряда я отдельно покажу:
- график;
- adf-проверку;
- acf/pacf;
- таблицу кандидатов;
- итоговый прогноз на 4 шага;
- сравнение с реальными значениями из test.

In [ ]:
series_summaries = []

for series_name in selected_series_names:
    info = yearly_series[series_name]
    start_year = info['start_year']

    train_values = pd.Series(
        info['train'].values,
        index=np.arange(start_year, start_year + len(info['train'])),
        name='train',
    )

    # по заданию прогнозирую только 4 будущих значения
    actual_future = pd.Series(
        info['test'].iloc[:4].values,
        index=np.arange(
            start_year + len(info['train']),
            start_year + len(info['train']) + 4,
        ),
        name='actual_future',
    )

    display(Markdown(f'## {series_name.lower()}'))

    plt.figure(figsize=(14, 4.5))
    plt.plot(
        train_values.index,
        train_values.values,
        marker='o',
        linewidth=2,
        color='steelblue',
        label='train часть',
    )
    plt.plot(
        actual_future.index,
        actual_future.values,
        marker='o',
        linewidth=2,
        color='black',
        label='реальные будущие значения',
    )
    plt.axvline(train_values.index[-1], color='gray', linestyle=':', linewidth=1.5)
    plt.title(f'{series_name.lower()}: исходный ряд и будущие 4 точки')
    plt.xlabel('год')
    plt.ylabel('значение')
    plt.legend()
    plt.show()

    adf_table = make_adf_table(train_values)
    display(adf_table)

    diff_series, used_d = choose_diff_series(train_values)
    print('для acf/pacf дальше удобнее смотреть ряд после d =', used_d)

    fig, axes = plt.subplots(3, 1, figsize=(14, 9))

    axes[0].plot(
        diff_series.index,
        diff_series.values,
        marker='o',
        linewidth=1.8,
        color='slateblue',
    )
    axes[0].axhline(0, color='black', linewidth=1)
    axes[0].set_title(f'{series_name.lower()}: ряд после differencing')
    axes[0].set_xlabel('год')

    acf_lags = min(10, len(diff_series) // 2 - 1)
    pacf_lags = min(10, len(diff_series) // 2 - 1)
    acf_lags = max(acf_lags, 1)
    pacf_lags = max(pacf_lags, 1)

    plot_acf(diff_series, lags=acf_lags, ax=axes[1])
    axes[1].set_title(f'{series_name.lower()}: acf')

    plot_pacf(diff_series, lags=pacf_lags, ax=axes[2], method='ywm')
    axes[2].set_title(f'{series_name.lower()}: pacf')

    plt.tight_layout()
    plt.show()

    candidate_table = evaluate_orders(train_values, actual_future)
    display(candidate_table.head(10))

    converged_table = candidate_table[candidate_table['сошлась']].reset_index(drop=True)
    selected_row = converged_table.iloc[0]
    selected_order = tuple(selected_row['order'])

    print('выбранная модель:', selected_order)

    selected_fit = ARIMA(
        train_values,
        order=selected_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit()

    forecast_values = selected_fit.forecast(steps=4)

    comparison_table = pd.DataFrame(
        {
            'год': actual_future.index,
            'факт': actual_future.values,
            'прогноз': forecast_values.values,
        }
    )
    comparison_table['ошибка'] = comparison_table['факт'] - comparison_table['прогноз']
    comparison_table['abs ошибка'] = np.abs(comparison_table['ошибка'])

    display(comparison_table.round(2))

    plt.figure(figsize=(14, 4.5))
    plt.plot(
        train_values.index,
        train_values.values,
        marker='o',
        linewidth=2,
        color='steelblue',
        label='train часть',
    )
    plt.plot(
        actual_future.index,
        actual_future.values,
        marker='o',
        linewidth=2,
        color='black',
        label='реальные значения',
    )
    plt.plot(
        actual_future.index,
        forecast_values.values,
        marker='o',
        linewidth=2,
        color='crimson',
        label='прогноз arima',
    )
    plt.axvline(train_values.index[-1], color='gray', linestyle=':', linewidth=1.5)
    plt.title(f'{series_name.lower()}: прогноз на 4 года')
    plt.xlabel('год')
    plt.ylabel('значение')
    plt.legend()
    plt.show()

    ljung_box_table = acorr_ljungbox(
        selected_fit.resid.dropna(),
        lags=[5, 10],
        return_df=True,
    ).reset_index().rename(
        columns={
            'index': 'лаг',
            'lb_stat': 'статистика ljung-box',
            'lb_pvalue': 'p-value',
        }
    )
    display(ljung_box_table)

    series_summaries.append(
        {
            'ряд': series_name,
            'выбранная модель': selected_order,
            'mae': selected_row['mae'],
            'rmse': selected_row['rmse'],
            'mape, %': selected_row['mape, %'],
            'aic': selected_row['aic'],
            'ljung-box p-value': selected_row['ljung-box p-value'],
        }
    )

summary_table = pd.DataFrame(series_summaries)
display(Markdown('## сводка по трем рядам'))
display(summary_table.round(3))

## вывод

по всем трем выбранным рядам модель `arima` удалось подобрать так, чтобы прогноз на 4 шага вперед был не просто формальным, а реально проверяемым на скрытой части m3.

что особенно видно по работе:
- для разных рядов подходят разные значения `d`;
- даже внутри одной yearly-группы динамика может быть очень разной;
- подбор через holdout из official test дает более честную картину, чем просто подгонка по истории.

для рядов `n 156` и `n 158` прогноз оказался особенно точным, а для `n 153` модель тоже работает, но с более заметной ошибкой.